# 📺 Yada Yada Data: Text, Ambiguity & Defensible Choices

### SIADS 505 · Other Assignments · Assignment 4
**100 points — 10 questions × 10 points each**

---

The first three assignments in this folder had right answers. This one is
different, and it is different on purpose.

You are handed 26,389 lines of dialogue from a nine-season sitcom, along with
its episode list and its character list. Half of the questions below have a
single correct answer and are graded that way. **The other half do not.** They
ask things like *"who is in this episode?"* and *"how many times does someone
say that?"* — questions that sound precise, and are not, because the data does
not contain a column called `is_in_this_episode` and never will.

For those questions the assignment is:

1. **Look at the data first.** Every underdetermined question has an ungraded
   exploration cell above it. Run it. Change it. The point is to find out what
   the choice actually costs before you make it.
2. **Write a function with the choice exposed as a keyword argument**, and make
   it work correctly for *every* setting of that argument — not just yours.
3. **Declare a default.** The default is your answer.
4. **Write down why**, in the `ASSUMPTIONS` dict below. A sentence or two: what
   you chose, why it is defensible *on this data*, and what it costs you.

The hidden checks never compare your default against a blessed one. They check
that the knob works at settings you did not pick, that your default is inside
the range of defensible options, and that you wrote something real in
`ASSUMPTIONS`. **A different choice, defended, scores the same.**

> **The corpus is simulated.** The show's structure — nine seasons, 180
> episodes — its characters, its writers and its directors are real. Every line
> of dialogue, every title, every air date and every audience figure was
> generated from a fixed seed. A handful of the show's famous catchphrases were
> sprinkled through the generated dialogue so Question 7 has something to find.
> Nothing else in here is a real quotation, and none of the numbers are real.

**Ground rules.** Every answer is a function that takes its data as arguments
and returns a value. Do not read files from inside a function unless the
question says to, do not modify the frame you were handed, and do not rename
the functions.


## The data

Three CSV files live in `data/`.

### `data/lines.csv` — one row per line of dialogue, 26,389 rows

| column | what it is |
|---|---|
| `line_id` | `L00001`, unique, in script order |
| `episode_id` | `S04E17`, joins to `episodes.csv` |
| `scene_no` | scene number within the episode |
| `line_no` | line number within the episode |
| `location` | where the scene is set |
| `speaker` | **as the script supervisor typed it** |
| `dialogue` | the line, sometimes with a stage direction in it |

`speaker` is the messy one. **234 distinct strings** in that column stand for
**89 actual speakers**: `JERRY`, `Jerry`, `jerry`, `JERRY:`, `  JERRY `,
`JERRY (V.O.)`, `KRAMER  (cont'd)`, `GEORGE [entering]`. A few lines are
credited to two people at once — `JERRY AND ELAINE`, `GEORGE & WAITRESS`.

`dialogue` carries stage directions inside round or square brackets, either as
the whole line (`(the door slams)`) or wrapped around it
(`(sighs) I can't believe you did that.`). **496 lines are nothing but a stage
direction.**

### `data/episodes.csv` — one row per episode, 180 rows

`episode_id`, `season` (1–9), `episode_in_season`, `episode_overall` (1–180),
`title`, `air_date`, `writers`, `director`, `runtime_min`,
`us_viewers_millions` (**14 missing**), `imdb_rating` (**6 missing**).

`writers` holds **one or two names separated by ` | `** — 65 of the 180
episodes are co-written. That is Question 8.

### `data/characters.csv` — one row per character, 34 rows

`character` (the canonical upper-case name, which is what Question 2 produces),
`full_name`, `role`, `first_season`.

`role` is one of `main` (4), `recurring` (14), `guest` (13) or **`mentioned`
(3)** — characters everybody talks about who never say a word. They are on the
list so that a careless merge has something to invent.

Note what is **not** in here: there is no table saying which characters appear
in which episode, no scene-level cast list, and no marker for a joint credit.
Several of the questions below are about that absence.


---

## Setup

Run this first.


In [ ]:
import re
from itertools import combinations

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

print(f"pandas {pd.__version__} · numpy {np.__version__}")
print("What's the deal with this dataset?")


---

## Your assumptions go here

Five of the ten questions ask you to make a call the data cannot make for you.
**Fill in an entry here for each one as you go.** The checks want a real
paragraph — at least 140 characters, and they will reject `TODO` and friends.

A good entry has three parts:

- **what you chose**, concretely (`min_lines=3`, not "a sensible threshold");
- **why it is defensible on *this* data** — point at something you saw in the
  exploration cell, not at a general principle;
- **what it costs you** — every choice here throws something away. Say what.

Nobody is grading whether your answer matches the reference. They are grading
whether you know what you did.


In [ ]:
ASSUMPTIONS = {
    # Question 5 — how many lines does a character need before you call them
    # part of the episode?
    "episode_cast": "",

    # Question 6 — how should the fourteen missing audience figures be filled?
    "filled_viewers": "",

    # Question 7 — does case matter? do word boundaries?
    "phrase_counts": "",

    # Question 8 — how should a co-written episode be credited?
    "writer_credits": "",

    # Question 10 — the assumptions behind your own investigation. (You can
    # leave this pointing at the INVESTIGATION dict at the bottom.)
    "my_investigation": "",
}


---

## First contact

Look at the thing before you write anything. Nothing here is graded — poke at
it, add cells, break it. The exploration cells later on assume you have already
done this once.


In [ ]:
lines_raw = pd.read_csv("data/lines.csv")
episodes_raw = pd.read_csv("data/episodes.csv")
characters_raw = pd.read_csv("data/characters.csv")

print(lines_raw.shape, episodes_raw.shape, characters_raw.shape)
display(lines_raw.head(6))

print("\nThe speaker column, as typed:")
print(lines_raw["speaker"].value_counts().head(12).to_string())
print(f"\n{lines_raw['speaker'].nunique()} distinct strings in that column.")


In [ ]:
# Dialogue with stage directions in it, and dialogue that is only a stage direction.
bracketed = lines_raw["dialogue"].str.contains(r"[\(\[]", regex=True)
only_direction = lines_raw["dialogue"].str.fullmatch(r"\s*[\(\[][^\)\]]*[\)\]]\s*")

print(f"lines containing a bracketed direction: {bracketed.sum()}")
print(f"lines that are nothing else:            {only_direction.sum()}")
display(lines_raw.loc[bracketed & ~only_direction, ["speaker", "dialogue"]].head(5))
display(lines_raw.loc[only_direction, ["speaker", "dialogue"]].head(3))


In [ ]:
# What is missing, and where.
display(episodes_raw.isna().sum().to_frame("missing").query("missing > 0"))
display(episodes_raw.groupby("season")["us_viewers_millions"].agg(["count", "median"]).round(2))
print("\nRoles on the character list:")
print(characters_raw["role"].value_counts().to_string())
print("\nCharacters who never say a word:")
print(characters_raw.loc[characters_raw["role"] == "mentioned", "character"].tolist())


---

## Question 1 — Roll the tape  ·  10 points

Three loaders. Nothing clever, but every question after this one reads from
what they return.

- **`load_episodes(path="data/episodes.csv")`** — parse **`air_date`** as a
  datetime, index by **`episode_id`**. **180 rows, 10 columns.**
- **`load_lines(path="data/lines.csv")`** — index by **`line_id`**, leave the
  columns in their original order. **26,389 rows, 6 columns.**
- **`load_characters(path="data/characters.csv")`** — read it as it is, with
  `character` still a **column**. **34 rows, 4 columns.**


In [ ]:
def load_episodes(path="data/episodes.csv"):
    """Load episodes.csv with a datetime `air_date`, indexed by `episode_id`."""
    # YOUR CODE HERE
    raise NotImplementedError()


def load_lines(path="data/lines.csv"):
    """Load lines.csv, indexed by `line_id`."""
    # YOUR CODE HERE
    raise NotImplementedError()


def load_characters(path="data/characters.csv"):
    """Load characters.csv as it is, `character` still a column."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
episodes = load_episodes()
lines = load_lines()
characters = load_characters()

assert episodes.shape == (180, 10), f"expected (180, 10), got {episodes.shape}"
assert episodes.index.name == "episode_id" and episodes.index.is_unique
assert pd.api.types.is_datetime64_any_dtype(episodes["air_date"])
assert episodes.groupby("season").size().to_dict() == {
    1: 5, 2: 12, 3: 23, 4: 24, 5: 22, 6: 24, 7: 24, 8: 22, 9: 24
}, "nine seasons, 180 episodes"

assert lines.shape == (26389, 6), f"expected (26389, 6), got {lines.shape}"
assert lines.index.name == "line_id"
assert list(lines.columns) == [
    "episode_id", "scene_no", "line_no", "location", "speaker", "dialogue"
]
assert set(lines["episode_id"]) <= set(episodes.index), "every line belongs to an episode"

assert characters.shape == (34, 4), f"expected (34, 4), got {characters.shape}"
assert "character" in characters.columns
assert characters["role"].value_counts().to_dict() == {
    "recurring": 14, "guest": 13, "main": 4, "mentioned": 3
}

print(f"📼 {len(lines):,} lines across {len(episodes)} episodes, "
      f"{lines['episode_id'].nunique()} of which have any dialogue at all.")
print("✅ Q1 public checks passed.")
episodes.head()


---

## Question 2 — Who said that?  ·  10 points

Every question about a character depends on knowing which rows are that
character. Right now `JERRY`, `Jerry`, `jerry`, `JERRY:`, `  JERRY ` and
`JERRY (V.O.)` are six different strings.

Write **`clean_speakers(lines)`** returning a Series named **`speaker_clean`**
on the same index, applying **exactly this rule, in this order**:

1. delete anything inside round or square brackets, **including the brackets**;
2. delete every colon;
3. collapse each run of whitespace to a single space;
4. strip the leading and trailing whitespace;
5. upper-case what is left.

`234` distinct raw strings become **`89`** distinct speakers. Step 3 is not
optional: `"KRAMER  (cont'd)"` leaves a double space behind once step 1 has
run, and skipping it leaves you with `"KRAMER "` — which is not `"KRAMER"`.

Note what this rule does **not** do: `"JERRY AND ELAINE"` stays a single
speaker called `JERRY AND ELAINE`. That is a wart, and it is deliberate. It
comes back in Question 5.

*Hint:* `.str.replace(pattern, " ", regex=True)` with
`r"[\(\[][^\)\]]*[\)\]]"` handles both bracket styles in one pass.


In [ ]:
def clean_speakers(lines):
    """One canonical, upper-case name per line of dialogue."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
speaker = clean_speakers(lines)

assert isinstance(speaker, pd.Series) and speaker.name == "speaker_clean"
assert len(speaker) == 26389 and speaker.index.equals(lines.index)
assert speaker.nunique() == 89, f"expected 89 distinct speakers, got {speaker.nunique()}"

assert (speaker == speaker.str.upper()).all(), "upper-case the result"
assert (speaker == speaker.str.strip()).all(), "strip the whitespace"
assert not speaker.str.contains("  ").any(), "collapse the runs of whitespace"
assert not speaker.str.contains(r"[\(\[\):]").any(), "the annotations should be gone"

assert speaker.value_counts().head(4).index.tolist() == ["JERRY", "GEORGE", "ELAINE", "KRAMER"]
assert speaker.value_counts()["JERRY"] == 8599

joint = speaker[speaker.str.contains(" AND | & ")]
print(f"🗣️  234 raw strings → {speaker.nunique()} speakers. "
      f"{len(joint)} lines are still credited to two people at once:")
print("   ", ", ".join(sorted(joint.unique())[:4]), "...")
print("✅ Q2 public checks passed.")
speaker.value_counts().head(8)


---

## Question 3 — How much did they say?  ·  10 points

A line count is a crude measure of how much somebody talks: `"Get OUT!"` and a
forty-word monologue both count once. Words are better, and words need a
definition.

Write **`line_word_counts(lines)`** returning an **integer** Series named
**`word_count`** on the same index, using **exactly this rule**:

1. delete anything inside round or square brackets (a stage direction is not
   speech);
2. count the matches of the regex **`[A-Za-z']+`** in what is left.

So `"(sighs) I can't believe you did that."` is **6** words — `sighs` does not
count, and `can't` counts once. A line that is nothing but a stage direction is
**0**, and there are **496** of those.

*Hint:* `.str.count(r"[A-Za-z']+")` counts non-overlapping matches without a
loop. The apostrophe inside the character class is what keeps `can't` in one
piece.


In [ ]:
def line_word_counts(lines):
    """Spoken words per line, stage directions excluded."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
words = line_word_counts(lines)

assert isinstance(words, pd.Series) and words.name == "word_count"
assert words.dtype.kind in "iu", f"whole numbers, got {words.dtype}"
assert len(words) == 26389 and words.index.equals(lines.index)

assert words.sum() == 229145
assert (words == 0).sum() == 496, "the direction-only lines have no spoken words"
assert words.max() == 15

sample = "(sighs) I can't believe you did that."
probe = pd.DataFrame({"dialogue": [sample, "(the door slams)"]})
assert list(line_word_counts(probe)) == [6, 0], "check the rule on these two by hand"

print(f"✍️  {words.sum():,} spoken words, {words.mean():.1f} per line.")
print("✅ Q3 public checks passed.")
words.head()


---

## Question 4 — The Big Four  ·  10 points

Write **`speaking_share(lines)`** returning a DataFrame with **one row per
episode**, indexed by `episode_id` (sorted, index named `episode_id`) and
columns **`["JERRY", "GEORGE", "ELAINE", "KRAMER"]`** in that order.

Each value is that character's **share of the lines in that episode**, rounded
to **3 decimals**, using the cleaned speaker names from Question 2.

Two rules, both stated so this question stays deterministic — and both worth
arguing with later:

- **The denominator is every line in the episode**, including guest lines and
  the joint-credited ones. So the four columns do not sum to 1.
- **A character with no lines in an episode is `0.0`, not `NaN`.** Elaine is
  not in season 1 at all; those five rows read `0.0`.

That second rule is a real choice, and it is arguably the wrong one — a `0.0`
for season 1 says "Elaine was there and said nothing", which is false. It is
stated here so everyone computes the same thing. If it bothers you, Question 10
is where to do something about it.

*Hint:* `groupby(["episode_id", speaker]).size().unstack(fill_value=0)` gets
you counts; `.div(totals, axis=0)` turns them into shares;
`.reindex(columns=[...])` fixes the order and covers a lead who never speaks in
the slice you were handed.


In [ ]:
def speaking_share(lines):
    """Each lead's share of the lines in every episode."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
share = speaking_share(lines)

assert list(share.columns) == ["JERRY", "GEORGE", "ELAINE", "KRAMER"]
assert share.index.name == "episode_id" and len(share) == 180
assert list(share.index) == sorted(share.index)

season_one = [e for e in share.index if e.startswith("S01")]
assert share.loc[season_one, "ELAINE"].notna().all(), "0.0, not NaN"
assert (share.loc[season_one, "ELAINE"] == 0.0).all(), "Elaine is not in season 1"

assert abs(share.loc["S01E01", "GEORGE"] - 0.466) < 0.001
assert abs(share["JERRY"].mean() - 0.325) < 0.001
assert (share.sum(axis=1) <= 1.002).all(), "guest lines are in the denominator too"
assert share.sum(axis=1).min() < 0.9, "some episodes give a real chunk of the night away"

print("📊 Mean share of the dialogue, across all 180 episodes:")
print(share.mean().round(3).to_string())
print(f"\nJerriest episode: {share['JERRY'].idxmax()} at {share['JERRY'].max():.1%}")
print("✅ Q4 public checks passed.")
share.head()


---

## Question 5 — Who is in this episode?  ·  10 points

> **This is the first question with no right answer.**

There is no table of who appears in which episode. The obvious proxy is "they
said something" — but a waitress with one line is not *in the episode* the way
Kramer is, and treating her as though she were takes the typical cast from four
people to nearly seven.

Write **`episode_cast(lines, min_lines=???)`** returning a DataFrame indexed by
`episode_id` (sorted, **all 180 episodes, always**) with:

| column | |
|---|---|
| `cast_size` | how many characters cleared the bar — an **integer** |
| `cast` | their names, **alphabetical, joined with `", "`** |

A character is in the cast if they speak **at least `min_lines` lines** in that
episode, counting cleaned speaker names from Question 2. An episode where
nobody clears the bar still gets a row: `cast_size` `0`, `cast` `""`.

**What is graded:**

1. the function is correct at `min_lines=1`, `3` and `12` — not just at yours;
2. your **default** is an integer between 1 and 10;
3. `ASSUMPTIONS["episode_cast"]` explains the choice.

**Things worth noticing before you choose.** 345 (episode, character) pairs
have exactly one line. And the cleaning rule from Question 2 leaves
`JERRY AND ELAINE` sitting in the cast list of some episodes as though it were
a person — at `min_lines=1` you will see it. Neither of those is a bug you have
to fix here; both are things worth saying out loud in `ASSUMPTIONS`.


**Look before you choose.** This cell is ungraded. Run it, and change it.


In [ ]:
speaker = clean_speakers(lines)
per_episode = lines.assign(who=speaker).groupby(["episode_id", "who"]).size()

print("How many lines does a character get in an episode they are 'in'?")
print(per_episode.value_counts().sort_index().head(10).to_string())
print(f"\n{(per_episode == 1).sum()} (episode, character) pairs have exactly one line.")

print("\nAverage cast size, by where you put the bar:")
for threshold in (1, 2, 3, 4, 6, 8, 12, 20):
    kept = per_episode[per_episode >= threshold].groupby("episode_id").size()
    print(f"   min_lines={threshold:>2}  ->  {kept.mean():.2f} characters per episode")

print("\nWho shows up if the bar is 1? (one episode)")
print(sorted(per_episode.loc["S04E01"].index.tolist()))


In [ ]:
# Replace None with the threshold you are prepared to defend.
def episode_cast(lines, min_lines=None):
    """Who is in each episode, at a threshold you choose and defend."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
# The graded checks call this at settings you did not pick, so test it that way.
for threshold in (1, 3, 12):
    table = episode_cast(lines, min_lines=threshold)
    assert list(table.columns) == ["cast_size", "cast"]
    assert table.index.name == "episode_id"
    assert len(table) == 180, "every episode gets a row, even at a high threshold"
    assert table["cast_size"].dtype.kind in "iu"

assert abs(episode_cast(lines, min_lines=1)["cast_size"].mean() - 6.567) < 0.01
assert abs(episode_cast(lines, min_lines=3)["cast_size"].mean() - 4.339) < 0.01
assert abs(episode_cast(lines, min_lines=12)["cast_size"].mean() - 3.839) < 0.01

assert episode_cast(lines, min_lines=3).loc["S01E01", "cast"] == "GEORGE, JERRY, KRAMER"
assert "NEIGHBOR" in episode_cast(lines, min_lines=1).loc["S01E01", "cast"]

import inspect
chosen = inspect.signature(episode_cast).parameters["min_lines"].default
assert isinstance(chosen, int), "give min_lines a default — the default is your answer"
assert 1 <= chosen <= 10, "below 1 is meaningless; above 10 drops people who are plainly there"
assert len(" ".join(ASSUMPTIONS["episode_cast"].split())) >= 140, "write down why"

print(f"🎬 You chose min_lines={chosen}. "
      f"Average cast size at that threshold: "
      f"{episode_cast(lines)['cast_size'].mean():.2f}")
print("✅ Q5 public checks passed.")
episode_cast(lines).head()


---

## Question 6 — Fourteen missing numbers  ·  10 points

> **No right answer here either.**

`us_viewers_millions` is blank for 14 of the 180 episodes. You cannot drop
them — they are real episodes — and you cannot leave them, because whatever
reads this next will not know what to do with a gap.

Write **`filled_viewers(episodes, method="???")`** returning a float Series
named **`us_viewers_millions`**, indexed exactly like `episodes`, with **no
missing values** and **every reported figure left exactly as it was**, rounded
to **3 decimals**.

It must support **all three** of these, by these exact names:

| `method` | fill each gap with |
|---|---|
| `"global_median"` | the median of every reported figure in the whole series |
| `"season_median"` | the median of the reported figures **in that episode's season** |
| `"neighbours"` | the **mean of the nearest reported figure before and after it in air order**; at the very start or end of the run, whichever one exists |

Anything else should raise a `ValueError`.

**What is graded:** all three methods compute correctly, reported episodes are
untouched, your default is one of the three, and
`ASSUMPTIONS["filled_viewers"]` explains it.

*Hint for `"neighbours"`:* sort by `episode_overall`, then `.ffill()` gives you
the previous reported value on every row and `.bfill()` gives you the next.
`pd.concat([...], axis=1).mean(axis=1)` averages them and quietly does the
right thing at the ends, because `.mean()` skips `NaN`.


**Look before you choose.** This cell is ungraded. Run it, and change it.


In [ ]:
viewers = episodes["us_viewers_millions"]
print(f"{viewers.isna().sum()} of {len(viewers)} episodes have no reported audience.\n")

print("Audience by season — this is the whole argument:")
print(episodes.groupby("season")["us_viewers_millions"]
      .agg(reported="count", missing=lambda s: s.isna().sum(), median="median")
      .round(2).to_string())

print("\nWhere the gaps are:")
print(episodes.loc[viewers.isna(), ["season", "episode_overall", "title"]].to_string())

print(f"\nSeries median: {viewers.median():.2f}")
print("Season medians:", episodes.groupby('season')['us_viewers_millions'].median().round(1).to_dict())
print("\nFilling a season 9 gap with the series median would be off by roughly "
      f"{episodes.query('season == 9')['us_viewers_millions'].median() - viewers.median():.1f} million.")


In [ ]:
# Replace None with one of: 'global_median', 'season_median', 'neighbours'.
def filled_viewers(episodes, method=None):
    """Audience figures with the fourteen gaps filled."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
for method in ("global_median", "season_median", "neighbours"):
    filled = filled_viewers(episodes, method=method)
    assert isinstance(filled, pd.Series) and filled.name == "us_viewers_millions"
    assert filled.index.equals(episodes.index)
    assert filled.isna().sum() == 0, f"{method}: gaps remain"
    reported = episodes["us_viewers_millions"].notna()
    assert np.allclose(
        filled[reported], episodes.loc[reported, "us_viewers_millions"]
    ), f"{method}: you changed a figure that was actually reported"

assert abs(filled_viewers(episodes, method="global_median").sum() - 4456.410) < 0.01
assert abs(filled_viewers(episodes, method="season_median").sum() - 4435.960) < 0.01
assert abs(filled_viewers(episodes, method="neighbours").sum() - 4438.985) < 0.01

try:
    filled_viewers(episodes, method="vibes")
except ValueError:
    pass
else:
    raise AssertionError("an unknown method should raise ValueError")

import inspect
chosen = inspect.signature(filled_viewers).parameters["method"].default
assert chosen in ("global_median", "season_median", "neighbours"), "declare a default"
assert len(" ".join(ASSUMPTIONS["filled_viewers"].split())) >= 140, "write down why"

gaps = episodes.index[episodes["us_viewers_millions"].isna()]
print(f"📺 You chose {chosen!r}. What the three methods do to the fourteen gaps:")
print(pd.DataFrame({
    m: filled_viewers(episodes, method=m).loc[gaps] for m in
    ("global_median", "season_median", "neighbours")
}).round(2).head(6).to_string())
print("✅ Q6 public checks passed.")


---

## Question 7 — Catchphrases  ·  10 points

> **No right answer, and this time the wrong ones are funny.**

Write **`phrase_counts(lines, phrases, case_sensitive=???, whole_words=???)`**
returning an **integer** Series named **`occurrences`**, indexed by the phrases
**in the order they were handed to you** (index named `phrase`).

Each value is the **total number of non-overlapping occurrences** of that
phrase across the whole corpus — a line containing it twice counts twice.
Search the dialogue **with stage directions removed**, exactly as in
Question 3.

| argument | when `True` |
|---|---|
| `case_sensitive` | `"Giddy up"` and `"giddy up"` are different phrases |
| `whole_words` | the phrase only matches at word boundaries |

**What is graded:** all four combinations of the two flags are right, a phrase
that never appears returns `0` (and keeps its row), both defaults are booleans,
and `ASSUMPTIONS["phrase_counts"]` explains them.

*Hint:* `re.escape(phrase)` first, so a phrase containing a `.` or a `?` does
not turn into a wildcard. Then wrap it in `\b...\b` if `whole_words`, and pass
`flags=re.IGNORECASE` if not `case_sensitive`.
`Series.str.count(pattern, flags=...)` does the counting.


**Look before you choose.** This cell is ungraded. Run it, and change it.


In [ ]:
spoken = lines["dialogue"].str.replace(r"[\(\[][^\)\]]*[\)\]]", " ", regex=True)

print("Same phrase, four ways of asking:")
for phrase in ["Giddy up", "her", "Newman", "yada yada"]:
    row = {}
    for case_sensitive in (True, False):
        for whole_words in (True, False):
            pattern = re.escape(phrase)
            if whole_words:
                pattern = rf"\b{pattern}\b"
            flags = 0 if case_sensitive else re.IGNORECASE
            row[f"cs={int(case_sensitive)},ww={int(whole_words)}"] = int(
                spoken.str.count(pattern, flags=flags).sum()
            )
    print(f"  {phrase!r:<14} {row}")

print("\nWhere the letters h-e-r actually turn up:")
print(spoken[spoken.str.contains("her", case=False)].head(4).to_string())

print("\nHow the corpus spells the catchphrases:")
for phrase in ["giddy up", "yada yada", "newman", "serenity now"]:
    seen = spoken[spoken.str.contains(phrase, case=False)].str.strip().value_counts().head(3)
    print(f"  {phrase!r}: {seen.index.tolist()}")


In [ ]:
# Replace both Nones with True or False.
def phrase_counts(lines, phrases, case_sensitive=None, whole_words=None):
    """How often each phrase is said across the whole corpus."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
probe = ["yada yada", "Giddy up", "her", "Newman", "serenity now", "soup nazi"]
counts = phrase_counts(lines, probe)

assert isinstance(counts, pd.Series) and counts.name == "occurrences"
assert counts.index.name == "phrase" and list(counts.index) == probe
assert counts.dtype.kind in "iu"
assert counts.loc["soup nazi"] == 0, "a phrase nobody says still gets a row, at 0"

# The two flags, all four ways.
assert phrase_counts(lines, ["Giddy up"], case_sensitive=False, whole_words=True).iloc[0] == 549
assert phrase_counts(lines, ["Giddy up"], case_sensitive=True, whole_words=True).iloc[0] == 514
assert phrase_counts(lines, ["her"], whole_words=True).iloc[0] == 0
assert phrase_counts(lines, ["her"], whole_words=False).iloc[0] == 2183

# A phrase with punctuation in it must not behave like a regex.
assert phrase_counts(lines, ["don't."], whole_words=False).iloc[0] >= 0

import inspect
params = inspect.signature(phrase_counts).parameters
assert isinstance(params["case_sensitive"].default, bool), "declare a default"
assert isinstance(params["whole_words"].default, bool), "declare a default"
assert len(" ".join(ASSUMPTIONS["phrase_counts"].split())) >= 140, "write down why"

print("🗨️  Under your defaults:")
print(phrase_counts(lines, probe).to_string())
print("\nThe word 'her' appears 0 times. The letters h-e-r appear in 2,183 lines.")
print("✅ Q7 public checks passed.")


---

## Question 8 — Who wrote it?  ·  10 points

> **No right answer. This one is about what a number is *for*.**

`writers` holds one or two names separated by `" | "`. **65 of the 180
episodes are co-written.** So: does a co-writer get a whole credit, or half of
one?

Write **`writer_credits(episodes, credit="???")`** returning a **float** Series
named **`credits`**, indexed by `writer` (index named `writer`, **sorted
alphabetically**), supporting both of:

| `credit` | each episode gives |
|---|---|
| `"full"` | **1.0** to every credited writer — totals **245** |
| `"split"` | **1/n** to each of its n writers — totals **180** |

rounded to 3 decimals. Anything else raises `ValueError`.

Neither is wrong. `"full"` answers *"how many episodes did this person work
on?"*; `"split"` answers *"how much of this show did this person write?"* —
and only the second gives you a column you can turn into a percentage, because
only the second conserves episodes.

**What is graded:** both rules compute correctly, your default is one of the
two, and `ASSUMPTIONS["writer_credits"]` says which question you decided the
table was for.

*Hint:* `.str.split("|")`, strip each name, then `.explode()`. The weight per
row is `1.0` or `1 / len(names)`; attach it before you group.


**Look before you choose.** This cell is ungraded. Run it, and change it.


In [ ]:
sizes = episodes["writers"].str.split("|").apply(len)
print(f"{(sizes == 2).sum()} of {len(episodes)} episodes are co-written.\n")

names = episodes["writers"].str.split("|").apply(lambda ns: [n.strip() for n in ns])
solo = names[sizes == 1].explode().value_counts()
paired = names[sizes == 2].explode().value_counts()

comparison = pd.DataFrame({"solo": solo, "co_written": paired}).fillna(0).astype(int)
comparison["always_alone"] = comparison["co_written"] == 0
print("Who works alone and who does not — this is why the rule matters:")
print(comparison.sort_values("solo", ascending=False).head(10).to_string())

print("\nUnder 'full' the credits total more than there are episodes:")
print(f"   full : {comparison[['solo', 'co_written']].to_numpy().sum()}")
print(f"   split: {len(episodes)}")


In [ ]:
# Replace None with 'full' or 'split'.
def writer_credits(episodes, credit=None):
    """Episode credits per writer."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
full = writer_credits(episodes, credit="full")
split = writer_credits(episodes, credit="split")

for out in (full, split):
    assert isinstance(out, pd.Series) and out.name == "credits"
    assert out.index.name == "writer"
    assert list(out.index) == sorted(out.index), "alphabetical"
    assert len(out) == 18, f"expected 18 credited writers, got {len(out)}"

assert abs(full.sum() - 245.0) < 0.01, "180 episodes, 65 of them co-written"
assert abs(split.sum() - 180.0) < 0.01, "'split' conserves episodes; that is the point"
assert full.loc["Larry David"] == 63.0 and split.loc["Larry David"] == 47.0

try:
    writer_credits(episodes, credit="whatever")
except ValueError:
    pass
else:
    raise AssertionError("an unknown credit rule should raise ValueError")

import inspect
chosen = inspect.signature(writer_credits).parameters["credit"].default
assert chosen in ("full", "split"), "declare a default"
assert len(" ".join(ASSUMPTIONS["writer_credits"].split())) >= 140, "write down why"

print(f"✍️  You chose {chosen!r}. The two rules, side by side:")
print(pd.DataFrame({"full": full, "split": split})
      .sort_values("split", ascending=False).head(6).to_string())
print("✅ Q8 public checks passed.")


---

## Question 9 — Who shares a scene with whom?  ·  10 points

Back to a question with one answer.

Write **`scene_pairs(lines, characters)`** returning a DataFrame with a
**MultiIndex `["character_a", "character_b"]`** and a single **integer** column
**`scenes`**.

- A **scene** is a `(episode_id, scene_no)` pair.
- Two characters share a scene if **both speak at least one line in it**.
- Only characters whose `role` in `characters.csv` is **`main` or
  `recurring`** count. Walk-ons and joint credits are out.
- Each pair appears **once**, with `character_a` alphabetically **before**
  `character_b`.
- Only pairs that share **at least one** scene appear at all.
- Sorted by **`scenes` descending**, then `character_a`, then `character_b`.

*Hint:* reduce to the distinct `(episode_id, scene_no, character)` rows first,
then either `itertools.combinations` over each scene's sorted cast, or a
self-merge of that table on `["episode_id", "scene_no"]` keeping the rows where
`character_a < character_b`.


In [ ]:
def scene_pairs(lines, characters):
    """How many scenes each pair of named characters shares."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
pairs = scene_pairs(lines, characters)

assert list(pairs.columns) == ["scenes"]
assert list(pairs.index.names) == ["character_a", "character_b"]
assert pairs["scenes"].dtype.kind in "iu" and (pairs["scenes"] > 0).all()
assert all(a < b for a, b in pairs.index), "each pair once, alphabetically ordered"

guests = set(characters.loc[characters["role"] == "guest", "character"])
appearing = {name for pair in pairs.index for name in pair}
assert not (appearing & guests), "walk-ons do not belong in here"
assert not any(" AND " in name or " & " in name for name in appearing), "nor do joint credits"

assert len(pairs) == 74
assert pairs.index[0] == ("GEORGE", "JERRY")
assert pairs.iloc[0, 0] == 1243
assert pairs["scenes"].is_monotonic_decreasing, "sort by scenes, descending"

print("🤝 Most scenes together:")
print(pairs.head(6).to_string())
print("\nRarest pairings that happened at all:")
print(pairs.tail(4).to_string())
print("✅ Q9 public checks passed.")


---

## Question 10 — Your own investigation  ·  10 points

> **This one has no answer at all, because you are writing the question.**

Ask something of this corpus that nobody assigned you, answer it, and defend
how you answered it.

**Two things to hand in.**

**1. Fill in the `INVESTIGATION` dict** in the cell below, with four keys:

| key | what goes in it |
|---|---|
| `question` | one sentence, ending in a question mark |
| `assumptions` | every judgement call you made, and what each one costs — at least 160 characters |
| `method` | how you computed it, in a couple of sentences |
| `finding` | what you actually found, including how confident you are — at least 160 characters |

**2. Write `my_investigation(episodes, lines, characters)`** returning the
table that supports your finding. It must:

- take all three frames as arguments and **compute from them** — the checks run
  it on a two-season subset and fail it if the answer does not change;
- return the **same thing twice** on the same input;
- return a real table: at least 2 rows and 1 column.

**What is graded:** the write-up is complete and specific, the function runs
and returns a real table, it is reproducible, and it computes rather than
recites. **Nobody is grading whether your finding is interesting.** They are
grading whether somebody else could check it.

**Questions this corpus can actually support** — take one or ignore all of
them:

- Does the show get **less dependent on Jerry** over nine seasons, and who
  picks up the slack?
- Where does the show happen? **Monk's and Jerry's apartment** dominate — does
  that change as the run goes on, and what happens when Pendant Publishing
  disappears after season 6?
- Do the four leads **talk in measurably different ways** — line length,
  questions per line, exclamations?
- Is there a **director effect**? The show changes directors between seasons 5
  and 6. Does anything else change with it?
- Do the **catchphrases** cluster in particular seasons, and does a phrase burn
  out after it lands?
- Which pairs of characters share **more scenes than you would expect** given
  how much each of them talks?
- Does the **share of an episode going to guests** predict anything about its
  audience or its rating?

Whatever you pick, the assumptions are the assignment. "I counted lines, not
words, because ___, which means ___ is invisible to this table" is the shape of
the answer.


In [ ]:
INVESTIGATION = {
    # One sentence. It has to end in a question mark.
    "question": "",

    # Every judgement call, and what each one costs. At least 160 characters.
    "assumptions": "",

    # How you computed it, in a couple of sentences.
    "method": "",

    # What you found, and how sure you are. At least 160 characters.
    "finding": "",
}


In [ ]:
def my_investigation(episodes, lines, characters):
    """Answer the question you wrote in INVESTIGATION above."""
    # YOUR CODE HERE
    raise NotImplementedError()


**Public checks — worth 0 points. Run them.**


In [ ]:
assert isinstance(INVESTIGATION, dict)
for key in ("question", "assumptions", "method", "finding"):
    assert key in INVESTIGATION, f"INVESTIGATION is missing {key!r}"
    assert isinstance(INVESTIGATION[key], str) and INVESTIGATION[key].strip(), f"{key} is empty"
assert " ".join(INVESTIGATION["question"].split()).endswith("?"), "ask an actual question"
assert len(" ".join(INVESTIGATION["assumptions"].split())) >= 160
assert len(" ".join(INVESTIGATION["finding"].split())) >= 160

result = my_investigation(episodes, lines, characters)
assert isinstance(result, pd.DataFrame), "return a DataFrame"
assert result.shape[0] >= 2 and result.shape[1] >= 1, f"a real table, got {result.shape}"
assert result.equals(my_investigation(episodes, lines, characters)), "same input, same answer"

# The graded checks also run this on two seasons. Make sure that works.
some_episodes = episodes[episodes["season"].isin([4, 7])]
some_lines = lines[lines["episode_id"].isin(set(some_episodes.index))]
partial = my_investigation(some_episodes, some_lines, characters)
assert isinstance(partial, pd.DataFrame) and len(partial) >= 1
assert not result.equals(partial), "compute from the data you are handed, do not recite"

print("🔍", " ".join(INVESTIGATION["question"].split()))
print()
print(result.to_string())
print("\n✅ Q10 public checks passed — that's all ten.")


---

# 🎉 Victory lap

That's 100 points. Everything below is **ungraded**. It is here because the
questions above left several stones unturned on purpose, and turning them over
is the point of the whole exercise.


In [ ]:
# The joint-credit problem, in full. Question 2's rule leaves these as though
# they were characters. How many lines are we talking about, and does it matter?
speaker = clean_speakers(lines)
joint = speaker[speaker.str.contains(" AND | & ")]

print(f"{len(joint)} lines ({len(joint) / len(lines):.2%} of the corpus) credited to two people.")
print(joint.value_counts().head(8).to_string())
print("\nIf you split them across both speakers, whose share moves the most?")


In [ ]:
# Question 4 said an absent character is 0.0. Here is the case against.
share = speaking_share(lines)
season = lines["episode_id"].str.slice(1, 3).astype(int)

by_season = share.join(pd.Series(season.groupby(lines["episode_id"]).first(), name="season"))
print(by_season.groupby("season")[["JERRY", "GEORGE", "ELAINE", "KRAMER"]].mean().round(3).to_string())
print("\nSeason 1's Elaine column is five zeroes. Any average that includes them is wrong.")


In [ ]:
# Where does the show happen, and does that change?
where = lines.join(episodes[["season"]], on="episode_id")
table = pd.crosstab(where["location"], where["season"], normalize="columns").round(3)
table.loc[table.mean(axis=1).sort_values(ascending=False).index].head(10)


In [ ]:
# Do the four leads talk differently? Three crude measures, one table.
words = line_word_counts(lines)
style = lines.assign(who=clean_speakers(lines), words=words)
style = style[style["who"].isin(["JERRY", "GEORGE", "ELAINE", "KRAMER"])]

style.groupby("who").agg(
    lines=("words", "size"),
    words_per_line=("words", "mean"),
    questions=("dialogue", lambda s: s.str.contains(r"\?").mean()),
    exclamations=("dialogue", lambda s: s.str.contains("!").mean()),
).round(3).loc[["JERRY", "GEORGE", "ELAINE", "KRAMER"]]


In [ ]:
# Who shares more scenes than their line counts alone would predict?
pairs = scene_pairs(lines, characters).reset_index()
speaker = clean_speakers(lines)
scenes_each = (
    lines.assign(who=speaker)[["episode_id", "scene_no", "who"]]
    .drop_duplicates().groupby("who").size()
)
total_scenes = lines[["episode_id", "scene_no"]].drop_duplicates().shape[0]

pairs["expected"] = (
    pairs["character_a"].map(scenes_each) * pairs["character_b"].map(scenes_each) / total_scenes
)
pairs["ratio"] = (pairs["scenes"] / pairs["expected"]).round(2)
pairs.sort_values("ratio", ascending=False).head(10)


In [ ]:
# A picture, if matplotlib is installed.
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib is not installed — try: pip install matplotlib")
else:
    share = speaking_share(lines)
    season = share.index.str.slice(1, 3).astype(int)
    by_season = share.groupby(season).mean()

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.stackplot(
        by_season.index,
        *[by_season[c] for c in ["JERRY", "GEORGE", "ELAINE", "KRAMER"]],
        labels=["Jerry", "George", "Elaine", "Kramer"],
        colors=["#2b4570", "#a8763e", "#7d4f6d", "#3f7d6b"],
        alpha=0.9,
    )
    ax.set_title("Simulated share of the dialogue, by season")
    ax.set_xlabel("season")
    ax.set_ylabel("share of lines")
    ax.set_ylim(0, 1)
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()


---

### Your turn, again

Six things this notebook decided for you. Any one of them is a better
investigation than the one you just wrote, if you disagree with it:

1. **A joint credit is a character.** `JERRY AND ELAINE` is in the cast list at
   `min_lines=1`. Fixing it means splitting one line across two speakers — but
   then the shares in Question 4 no longer sum the way they did. Which
   inconsistency do you prefer?
2. **A line is the unit.** Every count above treats `"Get OUT!"` and a
   forty-word speech as one thing each. Redo Question 4 in words. Does the
   ranking change?
3. **Absence is zero.** Season 1's Elaine column. Would `NaN` be better? What
   breaks downstream if you make that change?
4. **A scene is a `scene_no`.** Two characters in the same scene who never
   address each other still count as sharing it. There is no way to tell from
   this data. What would you need?
5. **Stage directions are noise.** They are stripped everywhere above. They are
   also the only record of who is physically present without speaking. What
   could you do with them?
6. **The corpus is simulated.** Everything you just found is a property of a
   random number generator with seed 1989. Which of your findings would you
   still expect to hold in a real script corpus, and why?

---

<div align="center">

**📺 That's it. No hugging, no learning.**

*Except the part where you wrote down your assumptions. Do that part.*

</div>
